        # ⛺ P2　營地任務：時間序列與基準模型
        **統計冒險之旅 2026**　｜　Day 3（09/24 四）🗻 預測之巔　｜　關卡　｜　🏅 100 XP

        📖 實作；資料：勇者咖啡每日營收
        　任何模型都要先贏過基準

        ### 🎯 這一關你會學到
        - 移動平均、星期效應、假日效應
- lag 特徵：shift(1)、shift(7)
- 三種基準（上週同日、7 日平均、線性趨勢）的 RMSE——任何模型都要先贏過基準

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "P2"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["P2-1", "P2-2", "P2-3", "P2-4", "P2-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""

class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_P2_1(run):
    out, ns = run()
    s = 抓變數(ns, "信義_MA7")
    ok, msg = 資料框像(s, 種類="Series")
    if not ok: return (False, msg)
    if int(s.notna().sum()) != 171: return (False, "rolling(7) 前 6 天會是 NaN，不要 dropna 也不要 min_periods。")
    return (約等於(抓變數(ns, "MA7最後"), 4644.286, 1.0), "MA7最後 = 信義_MA7.iloc[-1]。")
任務定義("P2-1", _check_P2_1, 提示="信義.rolling(7).mean()。")

def _check_P2_2(run):
    out, ns = run()
    s = 抓變數(ns, "星期效應")
    ok, msg = 資料框像(s, 列=7, 種類="Series")
    if not ok: return (False, msg)
    if str(抓變數(ns, "最旺的星期")) != "星期日": return (False, "最旺的星期 = 星期效應.idxmax()。")
    return (約等於(抓變數(ns, "假日效應"), 2297.680, 5.0), "假日效應 = 假日平均[1] - 假日平均[0]。")
任務定義("P2-2", _check_P2_2, 提示="idxmax() 回傳最大值的索引（星期幾）。")

def _check_P2_3(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "相關_lag7"), 0.5189, 0.01): return (False, "相關_lag7 = daily['營收'].corr(daily['lag7'])；lag7 要先 groupby('分店') 再 shift(7)。")
    d = 抓變數(ns, "有lag")
    ok, msg = 資料框像(d, 列=510, 含欄位=["lag1", "lag7", "MA7"])
    return (ok, msg or "有lag = daily.dropna(subset=['lag1', 'lag7', 'MA7'])。")
任務定義("P2-3", _check_P2_3, 提示="dropna(subset=[...])。")

def _check_P2_4(run):
    out, ns = run()
    b = 抓變數(ns, "基準RMSE", dict)
    for k, v in {"上週同日": 1468.250, "七日平均": 1511.056, "線性趨勢": 1362.151}.items():
        if k not in b: return (False, f"基準RMSE 缺少「{k}」。")
        if not 約等於(b[k], v, 相對=0.03): return (False, f"「{k}」的 RMSE 不對（應約 {v:.0f}）。")
    return (str(抓變數(ns, "最佳基準")) == "線性趨勢", "最佳基準 = RMSE 最小的鍵。")
任務定義("P2-4", _check_P2_4, 提示="七日平均的預測值就是 測試['MA7']。")

def _check_P2_5(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "模型RMSE"), 1128.433, 相對=0.08): return (False, "模型RMSE = rmse(測試['營收'], rf.predict(X_te))；要用 300 棵樹、random_state=42、只用訓練期 fit。")
    return (bool(抓變數(ns, "贏過基準")) == True, "贏過基準 = 模型RMSE < 基準RMSE[最佳基準]。")
任務定義("P2-5", _check_P2_5, 提示="rmse(測試['營收'], rf.predict(X_te))。")


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
daily = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.0/data/coffee_daily.csv")
daily["日期"] = pd.to_datetime(daily["日期"])
daily = daily.sort_values(["分店", "日期"]).reset_index(drop=True)
print(daily.shape, daily["日期"].min().date(), daily["日期"].max().date())
daily.head()

## ⛺ 營地任務：時間序列與基準模型
每日營收是**時間序列**：今天和昨天有關、和上週同一天更有關。做時間序列的三個習慣：
1. **先畫圖、先平滑**：7 日移動平均把「星期幾」的鋸齒磨掉，看得到趨勢。
2. **把時間變成特徵**：星期幾、假日、上一天（lag 1）、上週同日（lag 7）、過去 7 天平均。
3. **先做基準**：「猜上週同日」這種笨方法的 RMSE 是幾分？你的模型沒贏過它，就沒資格上線。

> ⚠️ 切分時間序列不能 `train_test_split` 隨機打亂——會用「未來」預測「過去」。永遠用**最後一段時間**當測試。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
信義 = daily[daily["分店"] == "信義店"].set_index("日期")["營收"]
ax = 信義.plot(alpha=.4, label="每日營收", figsize=(10, 3.5))
信義.rolling(7).mean().plot(ax=ax, label="7 日移動平均", linewidth=2)
ax.set_title("信義店每日營收"); ax.legend(); plt.show()

### 🎯 任務 P2-1　移動平均

對信義店的 `信義`（以日期為索引的營收 Series）算 7 日移動平均 `信義_MA7`（`rolling(7).mean()`），並取出最後一天的值 `MA7最後`。

**預期結果（範例）**
```
4644.3 ｜ 前 6 天沒有值： 6
```

In [ ]:
# 🎯 任務 P2-1　移動平均（請保留這一行）
信義 = daily[daily["分店"] == "信義店"].set_index("日期")["營收"]
信義_MA7 = ???
MA7最後 = float(信義_MA7.iloc[-1])
print(round(MA7最後, 1), "｜ 前 6 天沒有值：", 信義_MA7.isna().sum())

In [ ]:
檢查("P2-1")   # ◀ 執行這一格，看看任務 P2-1 有沒有過關

## 2-2　星期效應與假日效應
「週末生意比較好」是直覺，數字化才能放進模型。`groupby("星期")["營收"].mean()` 就是每個星期幾的平均營收。

In [ ]:
順序 = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六", "星期日"]
星期效應 = daily.groupby("星期")["營收"].mean().reindex(順序)
星期效應.plot(kind="bar", title="各星期平均營收", rot=0); plt.show()
print(daily.groupby("假日")["營收"].mean().round(0))

### 🎯 任務 P2-2　星期效應與假日效應

算出 `星期效應`（各星期平均營收的 Series）、`最旺的星期`（平均最高的那一天）、`假日效應`（假日平均 − 非假日平均）。

**預期結果（範例）**
```
星期日 5467 2298
```

In [ ]:
# 🎯 任務 P2-2　星期效應與假日效應（請保留這一行）
星期效應 = daily.groupby("星期")["營收"].mean()
最旺的星期 = ???
假日平均 = daily.groupby("假日")["營收"].mean()
假日效應 = float(假日平均[1] - 假日平均[0])
print(最旺的星期, round(星期效應.max()), round(假日效應))

In [ ]:
檢查("P2-2")   # ◀ 執行這一格，看看任務 P2-2 有沒有過關

## 2-3　lag 特徵：把昨天和上週搬到今天這一列
`shift(1)` 把整欄往下推一格 → 每一列的「昨天營收」；`shift(7)` → 上週同日。**一定要先 `groupby("分店")`**，否則會把 A 店的昨天推給 B 店。
過去 7 天平均：`shift(1).rolling(7).mean()`——先 shift 再 rolling，才不會把「今天」算進去（那是偷看答案）。

In [ ]:
daily["lag1"] = daily.groupby("分店")["營收"].shift(1)
daily["lag7"] = daily.groupby("分店")["營收"].shift(7)
daily["MA7"] = daily.groupby("分店")["營收"].transform(lambda s: s.shift(1).rolling(7).mean())
print(daily[["日期", "分店", "營收", "lag1", "lag7", "MA7"]].head(9))
print(daily[["營收", "lag1", "lag7", "MA7"]].corr().round(3))

### 🎯 任務 P2-3　lag 特徵與相關

確認 `daily` 已有 `lag1`、`lag7`、`MA7` 三欄（用 2-3 節的程式），算出營收與 lag7 的相關係數 `相關_lag7`；把三欄有 NaN 的列丟掉存成 `有lag`（應剩 510 筆）。

**預期結果（範例）**
```
0.519 510
```

In [ ]:
# 🎯 任務 P2-3　lag 特徵與相關（請保留這一行）
daily["lag1"] = daily.groupby("分店")["營收"].shift(1)
daily["lag7"] = daily.groupby("分店")["營收"].shift(7)
daily["MA7"] = daily.groupby("分店")["營收"].transform(lambda s: s.shift(1).rolling(7).mean())
相關_lag7 = float(daily["營收"].corr(daily["lag7"]))
有lag = ???
print(round(相關_lag7, 3), len(有lag))

In [ ]:
檢查("P2-3")   # ◀ 執行這一格，看看任務 P2-3 有沒有過關

## 2-4　三種笨基準
測試窗：**最後 7 天**（三家店共 21 筆），之前的都是訓練。三種不用學習的猜法：
1. **上週同日**：預測 = lag7
2. **七日平均**：預測 = MA7
3. **線性趨勢**：每家店各自用「第幾天」對營收做線性迴歸（只用訓練期），外推到測試週

每一種都算 RMSE。之後任何模型都要先贏過最好的那一個。

In [ ]:
起點 = daily["日期"].max() - pd.Timedelta(days=6)
訓練 = 有lag[有lag["日期"] < 起點]; 測試 = 有lag[有lag["日期"] >= 起點]
print(len(訓練), len(測試), 測試["日期"].min().date(), "～", 測試["日期"].max().date())
def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
print("上週同日 RMSE：", round(rmse(測試["營收"], 測試["lag7"]), 1))

### 🎯 任務 P2-4　三種基準的 RMSE

做出字典 `基準RMSE`，鍵為 `"上週同日"`、`"七日平均"`、`"線性趨勢"`；`最佳基準` 是 RMSE 最小的那個鍵。線性趨勢：對每家店，用訓練期的 `天數 = (日期 − 全部最早日期).days` 對營收做 `LinearRegression`，預測測試期。

**預期結果（範例）**
```
{'上週同日': 1468.2, '七日平均': 1511.1, '線性趨勢': 1362.2} → 線性趨勢
```

In [ ]:
# 🎯 任務 P2-4　三種基準的 RMSE（請保留這一行）
起點 = daily["日期"].max() - pd.Timedelta(days=6)
訓練 = 有lag[有lag["日期"] < 起點]; 測試 = 有lag[有lag["日期"] >= 起點]
def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
基準RMSE = {"上週同日": rmse(測試["營收"], 測試["lag7"]),
           "七日平均": ???}
最早 = daily["日期"].min()
趨勢預測 = pd.Series(index=測試.index, dtype=float)
for 店, g in 測試.groupby("分店"):
    tr = 訓練[訓練["分店"] == 店]
    lr = LinearRegression().fit((tr["日期"] - 最早).dt.days.values.reshape(-1, 1), tr["營收"])
    趨勢預測[g.index] = lr.predict((g["日期"] - 最早).dt.days.values.reshape(-1, 1))
基準RMSE["線性趨勢"] = rmse(測試["營收"], 趨勢預測)
最佳基準 = min(基準RMSE, key=基準RMSE.get)
print({k: round(v, 1) for k, v in 基準RMSE.items()}, "→", 最佳基準)

In [ ]:
檢查("P2-4")   # ◀ 執行這一格，看看任務 P2-4 有沒有過關

## 2-5　模型上場：有沒有贏過基準？
把「今天的條件」（星期、分店、天氣、假日、氣溫、促銷）加上「過去的營收」（lag1、lag7、MA7）一起丟進隨機森林。
只用訓練期 fit，在測試週算 RMSE，和最佳基準比一比。

### 🎯 任務 P2-5　贏過基準

用 `RandomForestRegressor(300, random_state=42)`，特徵為 `星期、分店、天氣`（one-hot）＋ `假日、氣溫、促銷、lag1、lag7、MA7`，訓練期 fit、測試週預測，算 `模型RMSE`；`贏過基準` 是布林值（模型RMSE < 最佳基準的 RMSE）。

**預期結果（範例）**
```
1128.4 vs 最佳基準 1362.2 → 贏過基準： True
```

In [ ]:
# 🎯 任務 P2-5　贏過基準（請保留這一行）
欄位 = ["星期", "分店", "天氣", "假日", "氣溫", "促銷", "lag1", "lag7", "MA7"]
X_tr = pd.get_dummies(訓練[欄位], columns=["星期", "分店", "天氣"]).astype(float)
X_te = pd.get_dummies(測試[欄位], columns=["星期", "分店", "天氣"]).astype(float).reindex(columns=X_tr.columns, fill_value=0)
rf = RandomForestRegressor(300, random_state=42).fit(X_tr, 訓練["營收"])
模型RMSE = ???
贏過基準 = bool(模型RMSE < 基準RMSE[最佳基準])
print(round(模型RMSE, 1), "vs 最佳基準", round(基準RMSE[最佳基準], 1), "→ 贏過基準：", 贏過基準)

In [ ]:
檢查("P2-5")   # ◀ 執行這一格，看看任務 P2-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把測試窗改成最後 14 天，結論一樣嗎？
2. 拿掉 `lag7` 再跑一次隨機森林，RMSE 掉多少？這告訴你哪個特徵最重要。
3. 用 `pd.get_dummies` 前先把 `星期` 轉成「是否週末」一個欄位，比 7 欄好還是差？

---
## 🔑 通關密語
　你已經會把時間變成特徵，也養成了「先做基準」的好習慣。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：⛺ P3 營地任務：特徵工程與洩漏陷阱** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.2.0/notebooks/P3_camp_features_leakage.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/